In [ ]:
import json
import random
import shutil
from pathlib import Path

SCRIPT_DIR = Path.cwd()
BASE_DIR = SCRIPT_DIR / "CrackMap"
RGB_DIR = BASE_DIR / "images"
JSON_DIR = BASE_DIR / "labels"
OUTPUT_DIR = SCRIPT_DIR / "CrackMap_YOLO"

TRAIN_RATIO = 0.8
RANDOM_SEED = 42
REUSE_EXISTING_SPLIT = True

VALID_IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
# ==========================================


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def build_image_dict(rgb_folder: Path):
    """
    扫描图片文件夹，建立忽略大小写的映射字典。
    key: 文件名(无后缀)的小写
    value: 真实文件名
    """
    img_dict = {}
    if not rgb_folder.exists():
        raise FileNotFoundError(f"找不到图片目录: {rgb_folder}")

    for file in sorted(rgb_folder.iterdir()):
        if not file.is_file():
            continue

        if file.suffix.lower() in VALID_IMAGE_EXTS:
            base_name = file.stem.lower()

            if base_name in img_dict:
                raise ValueError(
                    f"发现重复图片基名: '{base_name}'\n"
                    f"冲突文件: {img_dict[base_name]} 和 {file.name}\n"
                    f"请先清理数据集，保证同一stem唯一。"
                )

            img_dict[base_name] = file.name

    return img_dict


def find_corresponding_image(base_name, img_dict):
    """
    智能查找图片：兼容前导零、大小写后缀
    """
    possible_keys = [base_name.lower()]

    if base_name.isdigit():
        for length in range(4, 7):
            possible_keys.append(base_name.zfill(length).lower())

    for key in possible_keys:
        if key in img_dict:
            return RGB_DIR / img_dict[key]

    return None


def read_name_list(path: Path):
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def save_name_list(path: Path, names):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(names))


def save_split_meta(split_dir: Path, seed, train_ratio, total_count):
    meta = {
        "random_seed": seed,
        "train_ratio": train_ratio,
        "total_count": total_count
    }
    with open(split_dir / "split_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


def load_or_create_split(all_base_names, split_dir: Path, seed, train_ratio, reuse_existing=True):
    """
    载入或创建固定划分。
    若 train/val 清单已存在，则直接复用。
    """
    train_list_path = split_dir / "train_stems.txt"
    val_list_path = split_dir / "val_stems.txt"

    if reuse_existing and train_list_path.exists() and val_list_path.exists():
        train_names = read_name_list(train_list_path)
        val_names = read_name_list(val_list_path)

        if train_names is None or val_names is None:
            raise RuntimeError("已检测到 split 文件，但读取失败。")

        overlap = set(train_names) & set(val_names)
        if overlap:
            raise RuntimeError(f"train/val 存在重叠样本，示例: {list(overlap)[:5]}")

        current_set = set(all_base_names)
        split_set = set(train_names) | set(val_names)

        if current_set != split_set:
            missing_in_split = sorted(list(current_set - split_set))[:10]
            extra_in_split = sorted(list(split_set - current_set))[:10]
            raise RuntimeError(
                "当前数据集与已保存 split 不一致。\n"
                f"当前数据集样本数: {len(current_set)}\n"
                f"split 样本数: {len(split_set)}\n"
                f"split 缺失样本示例: {missing_in_split}\n"
                f"split 多余样本示例: {extra_in_split}\n"
                "如果你修改了数据集，请删除 OUTPUT_DIR/splits 后重新运行。"
            )

        return train_names, val_names, True

    names = sorted(all_base_names)
    rng = random.Random(seed)
    rng.shuffle(names)

    train_count = int(len(names) * train_ratio)
    train_names = names[:train_count]
    val_names = names[train_count:]

    ensure_dir(split_dir)
    save_name_list(train_list_path, train_names)
    save_name_list(val_list_path, val_names)
    save_split_meta(split_dir, seed, train_ratio, len(names))

    return train_names, val_names, False


def main():
    print("当前工作目录:", Path.cwd())
    print("图片目录:", RGB_DIR)
    print("标注目录:", JSON_DIR)

    if not RGB_DIR.exists() or not JSON_DIR.exists():
        print(f"错误: 找不到 {RGB_DIR} 或 {JSON_DIR}，请检查路径！")
        return

    # 创建输出目录
    dirs_to_create = [
        OUTPUT_DIR / "images" / "train",
        OUTPUT_DIR / "images" / "val",
        OUTPUT_DIR / "labels" / "train",
        OUTPUT_DIR / "labels" / "val",
        OUTPUT_DIR / "splits",
    ]
    for d in dirs_to_create:
        ensure_dir(d)

    split_dir = OUTPUT_DIR / "splits"

    # 1. 扫描图片
    print("\n正在扫描图片文件夹...")
    image_dict = build_image_dict(RGB_DIR)
    print(f"共发现 {len(image_dict)} 张图片。")

    # 2. 匹配 JSON 与图片
    all_classes = set()
    json_data_dict = {}
    missing_count = 0
    skipped_no_size = 0
    total_json_count = 0

    print("正在解析 JSON 文件并匹配图片...")

    for json_file in sorted(JSON_DIR.iterdir()):
        if not json_file.is_file() or json_file.suffix.lower() != ".json":
            continue

        total_json_count += 1
        base_name = json_file.stem
        img_path = find_corresponding_image(base_name, image_dict)

        if not img_path:
            if missing_count == 0:
                print("\n警告: 存在找不到图片的JSON文件（仅显示前几个）:")
            if missing_count < 5:
                print(f"  - 找不到 {base_name} 对应的图片")
            missing_count += 1
            continue

        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"警告: 读取失败 {json_file.name}, 错误: {e}")
            continue

        img_w = data.get("imageWidth")
        img_h = data.get("imageHeight")

        if not img_w or not img_h:
            skipped_no_size += 1
            continue

        for shape in data.get("shapes", []):
            label = shape.get("label")
            if label:
                all_classes.add(label)

        json_data_dict[base_name] = {
            "json_path": json_file,
            "img_path": img_path,
            "img_w": img_w,
            "img_h": img_h,
            "img_name": img_path.name
        }

    if missing_count > 0:
        print(f"  ... 共有 {missing_count} 个JSON文件找不到对应图片被跳过。")
    if skipped_no_size > 0:
        print(f"  ... 共有 {skipped_no_size} 个JSON文件缺少 imageWidth / imageHeight 被跳过。")

    if len(json_data_dict) == 0:
        print("错误: 没有找到任何有效的图像-标注对。")
        return

    print(f"成功匹配到 {len(json_data_dict)} 对图像-JSON。")

    # 3. 类别映射
    sorted_classes = sorted(list(all_classes))
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(sorted_classes)}

    classes_file = OUTPUT_DIR / "classes.txt"
    with open(classes_file, "w", encoding="utf-8") as f:
        f.write("\n".join(sorted_classes))

    print(f"\n类别文件已保存至 {classes_file}")
    print(f"类别列表: {sorted_classes}")

    # 4. 划分 train / val
    all_base_names = sorted(list(json_data_dict.keys()))
    train_names, val_names, reused = load_or_create_split(
        all_base_names=all_base_names,
        split_dir=split_dir,
        seed=RANDOM_SEED,
        train_ratio=TRAIN_RATIO,
        reuse_existing=REUSE_EXISTING_SPLIT
    )

    if reused:
        print("\n已检测到现有划分文件，直接复用，确保结果完全一致。")
    else:
        print("\n未检测到现有划分文件，已按随机种子生成新的固定划分。")

    print(f"数据划分完成: 总计 {len(all_base_names)} 对文件")
    print(f"  训练集: {len(train_names)}")
    print(f"  验证集: {len(val_names)}")

    # 5. 转成 YOLO 标签并复制图片
    print("\n正在生成 YOLO 格式文件和复制图片...")

    def process_split(names, split_type):
        label_dir = OUTPUT_DIR / "labels" / split_type
        image_dir = OUTPUT_DIR / "images" / split_type

        for name in names:
            data = json_data_dict[name]
            json_path = data["json_path"]
            img_w = data["img_w"]
            img_h = data["img_h"]
            img_name = data["img_name"]

            try:
                with open(json_path, "r", encoding="utf-8") as f:
                    raw_data = json.load(f)
            except Exception as e:
                print(f"警告: 无法读取 {json_path}，跳过。错误: {e}")
                continue

            final_lines = []

            for shape in raw_data.get("shapes", []):
                label = shape.get("label")
                points = shape.get("points")
                shape_type = shape.get("shape_type", "polygon")

                if not label or not points:
                    continue
                if label not in class_to_id:
                    continue

                # 计算外接矩形
                try:
                    if shape_type == "rectangle" and len(points) >= 2:
                        x1, y1 = points[0]
                        x2, y2 = points[1]
                    else:
                        xs = [p[0] for p in points]
                        ys = [p[1] for p in points]
                        x1, x2 = min(xs), max(xs)
                        y1, y2 = min(ys), max(ys)
                except Exception:
                    continue

                # YOLO 归一化
                x_center = ((x1 + x2) / 2.0) / img_w
                y_center = ((y1 + y2) / 2.0) / img_h
                width = abs(x2 - x1) / img_w
                height = abs(y2 - y1) / img_h

                # 裁剪到合理范围
                x_center = max(0.0, min(1.0, x_center))
                y_center = max(0.0, min(1.0, y_center))
                width = max(0.0, min(1.0, width))
                height = max(0.0, min(1.0, height))

                cls_id = class_to_id[label]
                final_lines.append(
                    f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                )

            # 保存 txt 标签
            label_out_path = label_dir / f"{name}.txt"
            with open(label_out_path, "w", encoding="utf-8") as f:
                f.write("\n".join(final_lines))

            # 复制图片
            img_out_path = image_dir / img_name
            shutil.copy2(data["img_path"], img_out_path)

    process_split(train_names, "train")
    process_split(val_names, "val")

    # 6. 保存图像路径清单
    train_txt = split_dir / "train.txt"
    val_txt = split_dir / "val.txt"

    with open(train_txt, "w", encoding="utf-8") as f:
        for name in train_names:
            img_name = json_data_dict[name]["img_name"]
            f.write(f"images/train/{img_name}\n")

    with open(val_txt, "w", encoding="utf-8") as f:
        for name in val_names:
            img_name = json_data_dict[name]["img_name"]
            f.write(f"images/val/{img_name}\n")

    print(f"\n已保存划分清单:")
    print(f"  - {split_dir / 'train_stems.txt'}")
    print(f"  - {split_dir / 'val_stems.txt'}")
    print(f"  - {train_txt}")
    print(f"  - {val_txt}")
    print(f"  - {split_dir / 'split_meta.json'}")

    print(f"\n✅ 转换完成！YOLO 数据集已保存至: {OUTPUT_DIR}")


# 直接运行
main()
